# THEREDA → ThermoHub Import

Imports a raw THEREDA JSON export into the ThermoHub-schema ArangoDB graph via the
`TheredaImportData` pipeline (`format="TheredaImport"`). This is the **import** half of the
THEREDA round-trip; the companion notebook `export-THEREDA-GEMS3-data.ipynb` (or the standalone
`export-gems3.py` script) exports the resulting graph back out to GEMS3 backup JSON.

**Pipeline:** raw THEREDA JSON → `TheredaImportData::split_thereda_data()` (splits into per-block
sibling files cached under `data-in/<year>/`, filters out phases/components with no real data) →
declarative `FormatStructDataFile` JSON scripts (`scripts-in/*.json`, one per block/vertex-label
pair, run through the vendored Lua engine) → `VertexElement`/`VertexSubstance`/`VertexReaction`/
`VertexPhase`/`VertexInteraction` documents written into ArangoDB.

Run cells top to bottom. Restart the kernel after any `thermomatch` C++ rebuild — a live kernel
keeps the previously-compiled extension in memory and won't pick up code changes.

In [1]:
import thermomatch as match
match.ThermoImpexGenerator.create_local_database_if_not_existent = True

`create_local_database_if_not_existent = True` lets the `ThermoImpexGenerator` constructor below
auto-create the target ArangoDB database if it doesn't exist yet — convenient for a first run
against a new database name, but also means a typo'd database name silently creates a new empty
database instead of raising an error.

In [ ]:
# Set path to `schemas` and `lua` not from the configuration file
match.ThermoImpexGenerator.setResourcesDirectory("../../Resources")

# Connect to Arango local database
generator = match.ThermoImpexGenerator("http://localhost:8529", "root", "", "ORD_THEREDA_2026-01", overwrite_existing=True, formats_from=match.ThermoImpexGenerator.File, split_to_blocks=False)

`setResourcesDirectory` points the C++ layer at `schemas/`/`lua/` (schema validation + the Lua
engine each `FormatStructDataFile` script runs through). `overwrite_existing=True` means re-running
this notebook against the same database name replaces existing documents rather than erroring on
duplicates. `split_to_blocks=False` refers to the generator's own top-level format list — THEREDA's
block-splitting still happens separately, inside `TheredaImportData::split_thereda_data()`, when
`runImport` is called a couple of cells below.

Note: this notebook does **not** currently call `match.loadGems3ExportRulesFile(...)` before
`runImport` (unlike `export-gems3.py`/the export notebook, which load it before their own calls).
Formula charge-fixing at import time (`fixImportedSubstanceFormulas`) consults the `RedoxElements`/
`FormulaOverrides` rules, so without loading a rules file here first, import-time formula fixes run
with an empty rule set.

In [3]:
# List of implemented thermodynamic formats
# print( generator.allFormats() )

# Execute import from PSI table format data
description = [
    match.ImpexRowDescription("bibliography", "VertexDataSource", "", "FormatStructDataFile", "scripts-in/Bibliography.Thereda.FormatStructDataFile.json"), # bibliography records
    match.ImpexRowDescription("elements", "VertexElement", "", "FormatStructDataFile", "scripts-in/Element.Thereda.FormatStructDataFile.json"), # element records
    match.ImpexRowDescription("substances", "VertexSubstance", "", "FormatStructDataFile", "scripts-in/Substance.Thereda.FormatStructDataFile.json"), # substance records
    match.ImpexRowDescription("reactions", "VertexSubstance", "", "FormatStructDataFile", "scripts-in/ReactionDependentSubstance.Thereda.FormatStructDataFile.json"), # reaction dependent substance records
    match.ImpexRowDescription("reactions", "VertexReaction", "group", "FormatStructDataFile", "scripts-in/Reaction.Thereda.FormatStructDataFile.json"), # reaction records
    match.ImpexRowDescription("reactions", "EdgeTakes", "group", "FormatStructDataFile", "scripts-in/ReactionEdges.Thereda.FormatStructDataFile.json"), # reaction edge records
    match.ImpexRowDescription("phases", "VertexPhase", "", "FormatStructDataFile", "scripts-in/Phase.Thereda.FormatStructDataFile.json"), # phase records 
]

Each `ImpexRowDescription(block_name, vertex_label, group, "FormatStructDataFile", script_path)`
pairs one raw-JSON block (as split out by `split_thereda_data()`) with the declarative script that
maps it into a vertex/edge collection.

**Order matters here**: the later `generateEdges()` step creates `EdgeBelongs(phase→thermodataset)`,
which depends on the `thermodataset` document already existing — that document is only created
while the `"group"`-tagged reaction rows run. So the reaction rows must precede the phase row in
this list, as they do below.

Interactions/mixing models (Pitzer/SIT/RK) are **not** driven by any script in this list — they're
built entirely in C++ (`import_interaction_phases()`), reading `InteractionPhases` directly from
the split raw data.

In [4]:
generator.runImport("TheredaImport", description, "data-in/2026/THEREDA_2026-01_JSON.json", "THEREDA2026");
#generator.runImport("TheredaImport", description, "data-in/2026/x/THEREDA_JSON_11_07_2025.json", "THEREDA2026");

[jsonio17] [info] VertexThermoDataSet loading collection: 8494, loading query: 1004
[2026-07-26 16:47:24.921] [thermomatch] [info]  0% Preparing files ...
[2026-07-26 16:47:25.888] [thermomatch] [info]  10% Importing data ...
[jsonio17] [info] VertexPhase loading collection: 5030, loading query: 1483
[2026-07-26 16:47:25.897] [thermomatch] [info]  10% Importing VertexDataSource ...
[jsonio17] [info] VertexDataSource loading collection: 4137, loading query: 898
[2026-07-26 16:47:26.286] [thermomatch] [info]  20% Importing VertexElement ...
[jsonio17] [info] VertexElement loading collection: 4909, loading query: 1415
[2026-07-26 16:47:26.368] [thermomatch] [info]  30% Importing VertexSubstance ...
[jsonio17] [info] VertexSubstance loading collection: 6039, loading query: 1944
[2026-07-26 16:47:26.533] [thermomatch] [info] Redox-capable elements used for formula charge auto-fix: [Fe, Mn, Cr, Co, Ni, Cu, Ti, V, Mo, W, Ac, Th, Pa, U, Np, Pu, Am, Cm, Bk, Cf, Ce, Eu, As, Sb, Sn, Pb, Se, Tc, H

`runImport("TheredaImport", description, <raw_json_path>, <rcsetSymbol>)` drives the whole
pipeline: block-split → per-row import through the shared `import_records_from_format` choke point
(which also runs formula charge-fixing on every `VertexSubstance` as it's created) →
`generateEdges()` (fills in `properties.substances`/`properties.thermodataset` per phase, wires
`EdgeBelongs`, then builds the Pitzer/SIT/RK interaction graph). `rcsetSymbol` (`"THEREDA2026"`
here) becomes each phase's `properties.thermodataset`.

**Next step**: run `export-THEREDA-GEMS3-data.ipynb` (or `export-gems3.py`) against this same
database name to produce GEMS3 backup JSON.